# Music Annotation Project V7.8: Rebuilt Independent-Data Builder

**Runtime:** CPU. Select **Runtime > Run all**.

This replaces the earlier provisional V7.8 builder. It uses the genuine aligned annotated page as reconstruction input, a dilation-2 removal mask, frozen V7.5 `geometry_t80` evidence/routing semantics, and V7.7 structural labels.

## Independent input

```text
MusicAnnotationProject/independent_structural_validation_v7_8_builder/
  clean/
  annotated/
  corrected_masks/
  score_manifest.csv
```

The clean and annotated scans must share filename stems. The corrected mask is the authoritative reviewed annotation mask for that page.

## Leakage controls

- Clean truth is used for registration and post-reconstruction scoring only.
- Geometry/routing predictors use the aligned observed page, removal mask and reconstruction artefacts, never truth metrics.
- Predictor-only data and input hashes are locked before structural outcomes are written.
- Structural truth is sealed separately until explicit release.
- This builder never reads the previous failed V7.8 independent outcome files.

## Controls

First run with all controls `False`. Inspect registration and predictor-schema artefacts only. Then create the lock. After registering the lock ID, build sealed outcomes. Finally, release the labelled dataset for the separate V7.8 validator.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-image


In [ ]:
import cv2,json,hashlib,math,inspect,shutil
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime,timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
INPUT_ROOT=ROOT/'independent_structural_validation_v7_8_builder'
CLEAN_DIR=INPUT_ROOT/'clean';ANNOTATED_DIR=INPUT_ROOT/'annotated';CORRECTED_MASK_DIR=INPUT_ROOT/'corrected_masks';MANIFEST_PATH=INPUT_ROOT/'score_manifest.csv'
OUTPUT_ROOT=ROOT/'independent_structural_validation_v7_8_rebuilt'
ALIGNED_DIR=OUTPUT_ROOT/'aligned_observed';RECON_DIR=OUTPUT_ROOT/'reconstruction_cache';BUILD_DIR=OUTPUT_ROOT/'builder';SEALED_DIR=OUTPUT_ROOT/'sealed';LOCKS_DIR=OUTPUT_ROOT/'locks';REVIEW_DIR=OUTPUT_ROOT/'registration_review'
for d in [CLEAN_DIR,ANNOTATED_DIR,CORRECTED_MASK_DIR,OUTPUT_ROOT,ALIGNED_DIR,RECON_DIR,BUILD_DIR,SEALED_DIR,LOCKS_DIR,REVIEW_DIR]:d.mkdir(parents=True,exist_ok=True)

CREATE_LOCK=False
EXPECTED_LOCK_ID=None
BUILD_SEALED_OUTCOMES=False
RELEASE_LABELLED_DATASET=False

# Authoritative fixed values from V7.1/V7.5/V7.7.
MASK_DILATION=2
CROP_MARGIN=48
MIN_COMPONENT_AREA=20
GEOMETRY_THRESHOLD=80
BAND_WIDTH=4
CONTEXT_RING_RADIUS=18
STRUCTURAL_GATES={'minimum_ink_recall':0.985,'minimum_ink_f1':0.95,'minimum_edge_f1':0.90,'maximum_removed_print_fraction':0.001,'maximum_component_mae':16.0}
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

CONFIG_PATH=ROOT/'project_config.json'
if not CONFIG_PATH.exists():raise RuntimeError(f'Missing registration config: {CONFIG_PATH}')
CONFIG=json.loads(CONFIG_PATH.read_text());REG=CONFIG['registration']

def image_map(folder):return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED}
def sha256_file(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
def object_hash(v):return hashlib.sha256(json.dumps(v,sort_keys=True,default=str).encode()).hexdigest()
def atomic_json(path,value):
 tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,default=str));tmp.replace(path)
def immutable(original,candidate,mask):
 out=original.copy();out[mask>0]=candidate[mask>0];assert np.array_equal(out[mask==0],original[mask==0]);return out


## Exact registration workflow from Notebook 01


In [ ]:
def align_pair(clean_path,annotated_path):
 clean=cv2.imread(str(clean_path),0);annotated=cv2.imread(str(annotated_path),0)
 if clean is None or annotated is None:raise ValueError('Unreadable image pair')
 cs=min(1.0,REG['max_feature_dimension']/max(clean.shape));ans=min(1.0,REG['max_feature_dimension']/max(annotated.shape))
 clean_small=cv2.resize(clean,None,fx=cs,fy=cs,interpolation=cv2.INTER_AREA);ann_small=cv2.resize(annotated,None,fx=ans,fy=ans,interpolation=cv2.INTER_AREA)
 clahe=cv2.createCLAHE(2.0,(8,8));sift=cv2.SIFT_create(nfeatures=REG['sift_features'],contrastThreshold=.015,edgeThreshold=15)
 ak,ad=sift.detectAndCompute(clahe.apply(ann_small),None);ck,cd=sift.detectAndCompute(clahe.apply(clean_small),None)
 if ad is None or cd is None:raise ValueError('SIFT descriptors missing')
 matcher=cv2.FlannBasedMatcher(dict(algorithm=1,trees=5),dict(checks=120));good=[]
 for pair in matcher.knnMatch(ad,cd,k=2):
  if len(pair)==2 and pair[0].distance<REG['ratio_test']*pair[1].distance:good.append(pair[0])
 if len(good)<20:raise ValueError(f'Only {len(good)} good matches')
 src=np.float32([ak[m.queryIdx].pt for m in good])/ans;dst=np.float32([ck[m.trainIdx].pt for m in good])/cs
 H,inliers=cv2.findHomography(src,dst,cv2.RANSAC,REG['ransac_threshold'],maxIters=15000,confidence=.999)
 if H is None:raise ValueError('Homography failed')
 aligned=cv2.warpPerspective(annotated,H,(clean.shape[1],clean.shape[0]),flags=cv2.INTER_CUBIC,borderValue=255)
 return clean,aligned,H,{'matches':len(good),'inliers':int(inliers.sum()),'inlier_percentage':round(100*int(inliers.sum())/len(good),2),'clean_shape':list(clean.shape),'annotated_shape':list(annotated.shape)}

def registration_preview(clean,aligned):
 p=np.zeros((*clean.shape,3),np.uint8);p[:,:,1]=clean;p[:,:,2]=aligned;return p


## Exact V7.1 dilation-2 mask and Telea cache


In [ ]:
def binary(mask):return np.where(mask>0,255,0).astype(np.uint8)
def prepare_mask(seed,dilation=MASK_DILATION):
 mask=binary(seed)
 if dilation>0:mask=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*dilation+1,2*dilation+1)))
 return mask

def padded_components(mask):
 n,lab,stats,_=cv2.connectedComponentsWithStats((mask>0).astype(np.uint8),8);h,w=mask.shape;boxes=[]
 for i in range(1,n):
  x,y,bw,bh,area=stats[i]
  if area>=8:boxes.append((max(0,x-CROP_MARGIN),max(0,y-CROP_MARGIN),min(w,x+bw+CROP_MARGIN),min(h,y+bh+CROP_MARGIN)))
 changed=True
 while changed:
  changed=False;merged=[]
  while boxes:
   ax0,ay0,ax1,ay1=boxes.pop();rest=[]
   for bx0,by0,bx1,by1 in boxes:
    if not (ax1<bx0 or bx1<ax0 or ay1<by0 or by1<ay0):ax0=min(ax0,bx0);ay0=min(ay0,by0);ax1=max(ax1,bx1);ay1=max(ay1,by1);changed=True
    else:rest.append((bx0,by0,bx1,by1))
   boxes=rest;merged.append((ax0,ay0,ax1,ay1))
  boxes=merged
 return boxes

def telea_reconstruct(observed,mask):
 result=observed.copy()
 for x0,y0,x1,y1 in padded_components(mask):
  im=observed[y0:y1,x0:x1];mk=mask[y0:y1,x0:x1];cand=cv2.inpaint(im,mk,3,cv2.INPAINT_TELEA);local=result[y0:y1,x0:x1];local[mk>0]=cand[mk>0]
 return immutable(observed,result,mask)


## Frozen V7.5 geometry_t80 and surrounding evidence


In [ ]:
def background_surface(observed,mask,sigma=45.0):
 visible=(mask==0)&(observed>=200);values=observed.astype(np.float32)*visible;weights=visible.astype(np.float32)
 num=cv2.GaussianBlur(values,(0,0),sigma);den=cv2.GaussianBlur(weights,(0,0),sigma);surface=num/np.maximum(den,1e-4);fallback=cv2.GaussianBlur(observed,(0,0),sigma)
 return np.clip(np.where(den>0.02,surface,fallback),0,255).astype(np.uint8)
def outside_dark(observed,mask):return (observed<190)&(mask==0)
def support_maps(observed,mask):
 dark=outside_dark(observed,mask);h=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((1,35),np.uint8))>0;v=cv2.morphologyEx(dark.astype(np.uint8)*255,cv2.MORPH_OPEN,np.ones((15,1),np.uint8))>0
 n,lab,stats,_=cv2.connectedComponentsWithStats(dark.astype(np.uint8),8);c=np.zeros_like(dark)
 for i in range(1,n):
  x,y,w,hg,a=stats[i]
  if 3<=a<=650 and w<=40 and hg<=40:c[lab==i]=True
 k=np.ones((2*BAND_WIDTH+1,2*BAND_WIDTH+1),np.uint8)
 return cv2.dilate(h.astype(np.uint8),k)>0,cv2.dilate(v.astype(np.uint8),k)>0,cv2.dilate(c.astype(np.uint8),k)>0

def geometry_t80(observed,mask,telea):
 paper=background_surface(observed,mask);normalised=cv2.divide(observed,np.maximum(paper,1),scale=255);horizontal,vertical,compact=support_maps(observed,mask);m=mask>0
 preserve=m&(horizontal|vertical|compact)&(observed<200)
 remove=m&~preserve&(observed>=GEOMETRY_THRESHOLD)&(normalised>=GEOMETRY_THRESHOLD)
 ambiguous=m&~preserve&~remove
 result=observed.copy();result[remove]=paper[remove];result[ambiguous]=telea[ambiguous];result[preserve]=observed[preserve]
 return immutable(observed,result,mask),{'preserve':preserve,'remove':remove,'ambiguous':ambiguous,'horizontal':horizontal,'vertical':vertical,'compact':compact,'paper':paper,'normalised':normalised}

def surrounding_features(observed,component,routes):
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max()+1;y0,y1=ys.min(),ys.max()+1;area=int(component.sum());w=x1-x0;h=y1-y0
 kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(2*CONTEXT_RING_RADIUS+1,2*CONTEXT_RING_RADIUS+1));ring=(cv2.dilate(component.astype(np.uint8),kernel)>0)&~component
 vals=observed[ring];edges=cv2.Canny(observed,70,170)>0;depth=cv2.distanceTransform(component.astype(np.uint8),cv2.DIST_L2,5)
 def f(k):return float(np.sum(routes[k]&component)/area)
 return {'w':w,'h':h,'area':area,'aspect_ratio':max(w,h)/max(min(w,h),1),'fill_ratio':area/max(w*h,1),'touches_page_border':bool(x0==0 or y0==0 or x1==observed.shape[1] or y1==observed.shape[0]),'context_pixels':int(ring.sum()),'context_edge_density':float(edges[ring].mean()) if ring.any() else 0.0,'context_dark_density':float((vals<190).mean()) if vals.size else 0.0,'context_mean_intensity':float(vals.mean()) if vals.size else 255.0,'context_intensity_std':float(vals.std()) if vals.size else 0.0,'maximum_mask_depth':float(depth[component].max()),'horizontal_fraction':f('horizontal'),'vertical_fraction':f('vertical'),'compact_fraction':f('compact'),'preserve_fraction':f('preserve'),'remove_fraction':f('remove'),'ambiguous_fraction':f('ambiguous'),'reconstruction_dependency_fraction':f('ambiguous'),'component_pixels':area}


## V7.7 structural truth


In [ ]:
def printed_ink(im):return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im):return cv2.Canny(im,70,170)>0
def tolerant_scores(ref,pred,region,tol=1):
 ref=ref&region;pred=pred&region
 if not ref.sum():return np.nan,np.nan,np.nan
 k=np.ones((2*tol+1,2*tol+1),np.uint8);precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1);recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum();return float(precision),float(recall),float(2*precision*recall/max(precision+recall,1e-12))
def structural_truth(clean,recon,component,routes):
 ci=printed_ink(clean);ri=printed_ink(recon);ip,ir,if1=tolerant_scores(ci,ri,component);ep,er,ef1=tolerant_scores(edges(clean),edges(recon),component);diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32));mae=float(diff[component].mean())
 preserve=routes['preserve']&component;removed=routes['remove']&component
 preserved_clean_print_precision=float(np.sum(preserve&ci)/max(preserve.sum(),1)) if preserve.any() else 1.0
 removed_clean_print_fraction=float(np.sum(removed&ci)/max(component.sum(),1))
 checks={'ink_recall':(not np.isfinite(ir)) or ir>=STRUCTURAL_GATES['minimum_ink_recall'],'ink_f1':(not np.isfinite(if1)) or if1>=STRUCTURAL_GATES['minimum_ink_f1'],'edge_f1':(not np.isfinite(ef1)) or ef1>=STRUCTURAL_GATES['minimum_edge_f1'],'removed_print':removed_clean_print_fraction<=STRUCTURAL_GATES['maximum_removed_print_fraction'],'component_mae':mae<=STRUCTURAL_GATES['maximum_component_mae']}
 failures=[k for k,v in checks.items() if not v]
 return {'structural_safe':not failures,'structural_outcome':'structural_safe' if not failures else 'structural_unsafe','structural_failure_reasons':'|'.join(failures),'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_precision':ep,'edge_recall':er,'edge_f1':ef1,'component_mae':mae,'removed_clean_print_fraction':removed_clean_print_fraction,'preserved_clean_print_precision':preserved_clean_print_precision}


## Validate input roster and build aligned observed pages


In [ ]:
manifest=pd.read_csv(MANIFEST_PATH)
required={'page','source_group','score_title','edition_or_source','acquisition_notes','excluded_from_development'}
if required-set(manifest):raise RuntimeError(f'Manifest missing: {sorted(required-set(manifest))}')
if manifest.page.duplicated().any():raise RuntimeError('Manifest page values must be unique.')
if not manifest.excluded_from_development.astype(str).str.lower().isin(['true','1','yes']).all():raise RuntimeError('Every page must declare excluded_from_development=True.')
cm=image_map(CLEAN_DIR);am=image_map(ANNOTATED_DIR);mm=image_map(CORRECTED_MASK_DIR);pages=sorted(set(cm)&set(am)&set(mm))
if set(manifest.page.astype(str))!=set(pages):raise RuntimeError('Manifest and clean/annotated/corrected-mask filename stems must agree exactly.')
meta=manifest.set_index('page');input_rows=[];runtime={}
for page in pages:
 clean,aligned,H,diagnostics=align_pair(cm[page],am[page]);seed=cv2.imread(str(mm[page]),0)
 if seed is None or seed.shape!=clean.shape:raise RuntimeError(f'{page}: corrected mask shape mismatch')
 cv2.imwrite(str(ALIGNED_DIR/f'{page}.png'),aligned);cv2.imwrite(str(REVIEW_DIR/f'{page}_registration.png'),registration_preview(clean,aligned));np.save(REVIEW_DIR/f'{page}_homography.npy',H);(REVIEW_DIR/f'{page}_registration.json').write_text(json.dumps(diagnostics,indent=2))
 mask=prepare_mask(seed,2);telea=telea_reconstruct(aligned,mask);recon,routes=geometry_t80(aligned,mask,telea)
 np.savez_compressed(RECON_DIR/f'{page}.npz',observed=aligned,mask=mask,telea=telea,geometry_t80=recon,**{f'g80_{k}':v.astype(np.uint8) for k,v in routes.items() if k in ['preserve','remove','ambiguous','horizontal','vertical','compact']})
 runtime[page]={'clean':clean,'observed':aligned,'mask':mask,'telea':telea,'recon':recon,'routes':routes}
 input_rows.append({'page':page,'source_group':str(meta.loc[page,'source_group']),'clean_sha256':sha256_file(cm[page]),'annotated_sha256':sha256_file(am[page]),'corrected_mask_sha256':sha256_file(mm[page]),'aligned_sha256':sha256_file(ALIGNED_DIR/f'{page}.png'),'registration_matches':diagnostics['matches'],'registration_inliers':diagnostics['inliers'],'registration_inlier_percentage':diagnostics['inlier_percentage'],'removal_mask_pixels':int((mask>0).sum())})
print('Prepared independent pages:',len(pages),'source groups:',manifest.source_group.nunique())


## Build predictor-only component dataset


In [ ]:
pred_rows=[]
for page in pages:
 z=runtime[page];n,lab,stats,_=cv2.connectedComponentsWithStats((z['mask']>0).astype(np.uint8),8)
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  comp=lab==cid;row={'source_group':str(meta.loc[page,'source_group']),'page':page,'component':cid};row.update(surrounding_features(z['observed'],comp,z['routes']));pred_rows.append(row)
predictors=pd.DataFrame(pred_rows).sort_values(['source_group','page','component']).reset_index(drop=True)
PREDICTOR_PATH=BUILD_DIR/'component_predictors_locked_candidate.csv';INPUT_PATH=BUILD_DIR/'input_manifest_locked_candidate.csv';predictors.to_csv(PREDICTOR_PATH,index=False);pd.DataFrame(input_rows).merge(manifest,on=['page','source_group'],validate='one_to_one').to_csv(INPUT_PATH,index=False)
print('Predictor components:',len(predictors));print('No structural outcomes calculated in this phase.')


## Lock predictors, settings and inputs


In [ ]:
SETTINGS={'schema':'v7.8-rebuilt-1','registration_config':REG,'mask_dilation':2,'crop_margin':CROP_MARGIN,'minimum_component_area':MIN_COMPONENT_AREA,'geometry_threshold':GEOMETRY_THRESHOLD,'band_width':BAND_WIDTH,'context_ring_radius':CONTEXT_RING_RADIUS,'structural_gates':STRUCTURAL_GATES}
lock_payload={'settings':SETTINGS,'settings_sha256':object_hash(SETTINGS),'predictor_sha256':sha256_file(PREDICTOR_PATH),'input_manifest_sha256':sha256_file(INPUT_PATH),'score_manifest_sha256':sha256_file(MANIFEST_PATH),'components':len(predictors),'pages':len(pages),'source_groups':int(manifest.source_group.nunique()),'reconstruction_cache_hashes':{p:sha256_file(RECON_DIR/f'{p}.npz') for p in pages}}
LOCK_ID=object_hash(lock_payload);LOCK_PATH=LOCKS_DIR/f'{LOCK_ID}.json'
if CREATE_LOCK:
 atomic_json(LOCK_PATH,{**lock_payload,'lock_id':LOCK_ID,'created_utc':datetime.now(timezone.utc).isoformat(),'status':'LOCKED_BEFORE_STRUCTURAL_OUTCOMES'});print('DATASET LOCK CREATED:',LOCK_ID)
else:print('CANDIDATE LOCK ID:',LOCK_ID);print('Set CREATE_LOCK=True only after registration previews, source roster and predictor schema are final.')


## Build sealed V7.7 structural outcomes after lock


In [ ]:
if BUILD_SEALED_OUTCOMES:
 if EXPECTED_LOCK_ID!=LOCK_ID or not LOCK_PATH.exists():raise RuntimeError('Structural outcomes require the exact registered lock ID.')
 lock=json.loads(LOCK_PATH.read_text())
 if lock['predictor_sha256']!=sha256_file(PREDICTOR_PATH) or lock['input_manifest_sha256']!=sha256_file(INPUT_PATH):raise RuntimeError('Locked predictor/input files changed.')
 truth=[]
 for page in pages:
  z=runtime[page];n,lab,stats,_=cv2.connectedComponentsWithStats((z['mask']>0).astype(np.uint8),8)
  for cid in predictors.loc[predictors.page==page,'component'].astype(int):
   row={'page':page,'component':cid};row.update(structural_truth(z['clean'],z['recon'],lab==cid,z['routes']));truth.append(row)
 truth_df=pd.DataFrame(truth).sort_values(['page','component']);SEALED_PATH=SEALED_DIR/f'{LOCK_ID}_structural_truth.csv';truth_df.to_csv(SEALED_PATH,index=False);atomic_json(SEALED_DIR/f'{LOCK_ID}_seal.json',{'lock_id':LOCK_ID,'truth_sha256':sha256_file(SEALED_PATH),'rows':len(truth_df),'outcomes_not_printed':True,'created_utc':datetime.now(timezone.utc).isoformat()});print('SEALED STRUCTURAL OUTCOMES CREATED. Counts intentionally suppressed.')
else:print('Structural outcomes remain sealed/unbuilt.')


## Explicit release for V7.8 validation


In [ ]:
if RELEASE_LABELLED_DATASET:
 if EXPECTED_LOCK_ID!=LOCK_ID:raise RuntimeError('Release requires the exact registered lock ID.')
 SEALED_PATH=SEALED_DIR/f'{LOCK_ID}_structural_truth.csv';SEAL_PATH=SEALED_DIR/f'{LOCK_ID}_seal.json'
 if not all(p.exists() for p in [LOCK_PATH,SEALED_PATH,SEAL_PATH]):raise RuntimeError('Lock or sealed truth missing.')
 seal=json.loads(SEAL_PATH.read_text())
 if seal['truth_sha256']!=sha256_file(SEALED_PATH):raise RuntimeError('Sealed truth hash mismatch.')
 labelled=predictors.merge(pd.read_csv(SEALED_PATH),on=['page','component'],validate='one_to_one');labelled.to_csv(OUTPUT_ROOT/'component_dataset.csv',index=False);manifest.to_csv(OUTPUT_ROOT/'score_manifest.csv',index=False);atomic_json(OUTPUT_ROOT/'dataset_release.json',{'lock_id':LOCK_ID,'component_dataset_sha256':sha256_file(OUTPUT_ROOT/'component_dataset.csv'),'score_manifest_sha256':sha256_file(OUTPUT_ROOT/'score_manifest.csv'),'settings_sha256':object_hash(SETTINGS),'released_utc':datetime.now(timezone.utc).isoformat(),'supersedes_provisional_v7_8_builder':True});print('LABELLED DATASET RELEASED FOR V7.8 VALIDATION');print('Lock ID:',LOCK_ID)
else:print('Labelled dataset remains unreleased.')
